# Segment 4: spike-resistant harmonic traversal regression

**Goal:** learn only recurring daily/weekly traversal patterns for Route `97-KEL`, direction 0, **Hwy 97 at Westside Rd Stn (103745) -> Queensway Exchange Bay B (102853)**. Spikes remain in the observations and residuals for a separate model; no congestion state, trend, lagged target, dwell or endpoint delay enters this model.

This follows [the traversal-time EDA](segment_4_traversal_time_eda.ipynb), but does not reuse its full-sample centring or outlier thresholds. It runs independently and offline from the trusted two-month snapshot, using the `transit_env` kernel from either the repository root or `notebooks/`.

**Interpretation:** this is a robust typical-duration component, not a forecast of incidents or an unbiased arithmetic mean. Final realtime endpoint times are reference proxies. The EDA already examined this cache, so the final week is an exploratory temporal evaluation, not a genuinely blind future test.

In [ ]:
from pathlib import Path
import json
import sys

repo_root = next((p for p in (Path.cwd().resolve(), Path.cwd().resolve().parent)
                  if (p / 'forecasting' / 'forecasting').is_dir()), None)
if repo_root is None:
    raise FileNotFoundError('Run from the repository root or notebooks directory.')
sys.path.insert(0, str(repo_root / 'forecasting'))

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display
from sklearn.base import clone
from sklearn.linear_model import Lasso
from sklearn.metrics import mean_absolute_error, median_absolute_error, mean_squared_error
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from threadpoolctl import threadpool_limits
from forecasting import (
    CalendarHarmonics, HuberLassoRegressor, build_route_segments, get_backtest_dataset,
    prepare_segment_traversal_eda, make_harmonic_time_splits, save_harmonic_experiment,
    prepare_early_traversal_comparison,
    make_traversal_comparison_timeseries, make_traversal_prediction_scatter,
)

## 1. Predeclare the calendar and robustness controls

The custom scikit-learn `CalendarHarmonics` transformer emits local daily and weekly sine/cosine pairs and their products. Products allow daily peak shape to vary smoothly through the week. Orders stay small because there are only about eight repetitions of each weekday. All features repeat exactly after one **local calendar week**; none represents an absolute date or long-run drift. UTC preserves event ordering; local wall time defines seasonality, including identical features for repeated DST clock times.

A training-fitted `StandardScaler` precedes coefficient shrinkage. `HuberLassoRegressor` repeatedly fits weighted sklearn `Lasso`, optimizing Huber loss with an L1 penalty. Its residual derivative is bounded by a **fixed** `huber_delta`; it does not estimate a scale that spikes could inflate. No percentile clipping or target-based row removal is used.

$$
\hat\beta = \arg\min_{\beta_0,\beta}\left[\frac{1}{n}\sum_i \rho_\delta\left(\log T_i-\beta_0-\phi(t_i)^\top\beta\right)+\alpha\lVert\beta\rVert_1\right].
$$

Here $T_i$ is traversal in minutes, $t_i$ is upstream departure, and $\phi$ contains calendar harmonics. Huber loss is $\rho_\delta(r)=r^2/2$ for $|r|\leq\delta$ and $\delta(|r|-\delta/2)$ otherwise. A residual transition of 0.15 log units is approximately a 16% multiplicative error. This is a robustness control, **not an incident definition**. The final weights are $\min(1,\delta/|r_i|)$ and remain positive.

Ordinary squared-loss Lasso on the **same log target and candidate features** is the comparator. Exponentiation enforces positive predictions but does not apply a mean/smearing correction that would reintroduce spike influence.

In [ ]:
dataset_id = '2026-11-10_2_months'
route_id, direction_id, segment_id = '97-KEL', 0, 4
expected_stop_pair = ('103745', '102853')
calendar_timezone = 'America/Vancouver'
service_day_start_hour = 3
n_splits, block_days, gap_days, min_train_days = 2, 7, 1, 28
harmonic_orders = [(2, 1), (4, 2), (6, 3)]
alpha_grid = [0.0001, 0.001, 0.003, 0.01]
huber_delta = 0.15
minimum_profile_visits = 10
representative_monday = '2026-11-02'
spike_fraction, spike_multiplier = 0.05, 8.0
random_state = 42
output_root = repo_root / 'data' / 'segment_harmonic_models'
figure_output_dir = repo_root / 'notebooks' / 'visuals' / 'segment_4_harmonic_traversal_regression'
figures = {}

In [ ]:
dataset = get_backtest_dataset(repo_root / 'data', source='cache', dataset_id=dataset_id)
static_dir = dataset.static_data_dir
manifest = json.loads((dataset.directory / 'metadata.json').read_text())
stops = pd.read_csv(static_dir / 'stops.txt', dtype={'stop_id': 'string'})
stop_times = pd.read_csv(static_dir / 'stop_times.txt', dtype={'trip_id': 'string', 'stop_id': 'string'})
trips = pd.read_csv(static_dir / 'trips.txt', dtype={'trip_id': 'string', 'route_id': 'string', 'shape_id': 'string'})
shapes = pd.read_csv(static_dir / 'shapes.txt', dtype={'shape_id': 'string'})
segments = build_route_segments(stops, stop_times, trips, shapes,
                                route_id=route_id, trip_direction_id=direction_id)
selected_segment = segments.set_index('segment_id').loc[segment_id]
resolved_pair = (str(selected_segment.from_stop_id), str(selected_segment.to_stop_id))
if resolved_pair != expected_stop_pair:
    raise ValueError(f'Segment identity changed: {resolved_pair}; review the controls.')
display(selected_segment.drop(labels=['geometry']).to_frame('reference segment'))
display(pd.Series({k: manifest[k] for k in ('dataset_id', 'trip_rows', 'trip_start_utc', 'trip_end_utc')}).to_frame('snapshot'))

## 2. Audit individual traversals and build temporal splits

Use one row per completed exact directed stop-pair visit within a reliably dated trip instance and consecutive numeric realtime stop sequences. Historical trip IDs need not match bundled GTFS. Historical shape is unconfirmed. Missing dwell or arrival delays do not exclude a valid traversal. Road duration is downstream arrival **minus upstream departure**, excluding upstream dwell.

Drop only invalid timing and the two partial recording-boundary service days, never valid high durations. The local 03:00-cutover date is a **service-day proxy**, not a recovered historical GTFS date. The split endpoint comes from recording bounds, so missing calendar dates cannot silently disappear.

Two expanding seven-day development blocks select complexity and alpha using **log-target MAE** (absolute relative/log error, rather than raw squared error). The last seven-day block is reserved. All scaling and robust fitting occur separately inside each training fold, with a one-day embargo and whole-journey purging when either endpoint's final publication or traversal completion is not before the fit cutoff.

This is retrospective calendar-component validation at final reported upstream departure, **not an operational as-of replay**. Even final publication is only a conservative availability proxy, not guaranteed label finalization. Deployment needs an independently available departure/decision timestamp and finalization policy.

In [ ]:
timing = prepare_segment_traversal_eda(dataset.trip_updates, selected_segment, route_id=route_id)
visits = timing.observations.query("traversal_exclusion_reason == 'included'").copy()
visits = visits.sort_values('upstream_departure_time').reset_index(drop=True)
local_clock = visits.upstream_departure_time.dt.tz_convert(calendar_timezone).dt.tz_localize(None)
visits['service_day'] = (local_clock - pd.Timedelta(hours=service_day_start_hour)).dt.normalize()
visits['weekday'], visits['hour'] = local_clock.dt.dayofweek, local_clock.dt.hour
visits['traversal_minutes'] = visits.traversal_seconds / 60
visits['available_at'] = visits[['feed_timestamp', 'upstream_feed_timestamp']].max(axis=1)
recording_bounds = pd.to_datetime([manifest['trip_start_utc'], manifest['trip_end_utc']], utc=True)
recording_days = (recording_bounds.tz_convert(calendar_timezone).tz_localize(None)
                  - pd.Timedelta(hours=service_day_start_hour)).normalize()
full_days = pd.date_range(recording_days[0] + pd.Timedelta(days=1),
                          recording_days[1] - pd.Timedelta(days=1), freq='D')
boundary_rows = int((~visits.service_day.isin(full_days)).sum())
visits = visits.loc[visits.service_day.isin(full_days)].reset_index(drop=True)
coverage = visits.groupby('service_day').size().reindex(full_days, fill_value=0).rename('visits')
display(timing.audit)
display(coverage.to_frame())
print(f'{len(visits):,} completed visits; {boundary_rows:,} partial-boundary visits excluded.')

In [ ]:
splits, split_summary = make_harmonic_time_splits(
    visits, evaluation_end=recording_days[1], n_splits=n_splits, block_days=block_days,
    gap_days=gap_days, min_train_days=min_train_days, timezone=calendar_timezone,
    service_day_start_hour=service_day_start_hour,
)
development_splits = splits[:-1]
train_index, test_index = splits[-1]
X = visits[['upstream_departure_time']]
y_log = np.log(visits.traversal_minutes.to_numpy())
X_train, X_test = X.iloc[train_index], X.iloc[test_index]
y_train_log = y_log[train_index]
display(split_summary)

## 3. Select and fit the calendar models

The same blocked folds and log-MAE score are used for both estimators. `refit=False` prevents automatic full-frame fitting (the frame includes reserved rows only for positional indexing); only the explicitly allowed final training indices are used after selection. The final block does not select parameters or trigger early stopping. Computation uses one search worker and one BLAS thread to avoid monopolizing the shared environment.

A global training median is a simple benchmark. There is deliberately no autoregressive residual, slope, changepoint or holiday effect. Repeated rush-hour peaks can be learned if supported across weeks, while isolated large residuals receive reduced fitting weight. A persistently recurring slowdown can still become seasonality; calendar-only data cannot establish the physical cause or perfectly separate all incidents.

In [ ]:
candidate_grid = [
    {'calendar__daily_order': [daily], 'calendar__weekly_order': [weekly],
     'regressor__alpha': alpha_grid}
    for daily, weekly in harmonic_orders
]
estimators = {
    'ordinary_lasso': Lasso(max_iter=20000, tol=1e-7),
    'robust_huber_lasso': HuberLassoRegressor(delta=huber_delta),
}
models, searches, candidate_tables = {}, {}, []
with threadpool_limits(limits=1):
    for name, estimator in estimators.items():
        pipeline = Pipeline([
            ('calendar', CalendarHarmonics(timezone=calendar_timezone)),
            ('scale', StandardScaler()), ('regressor', estimator),
        ])
        search = GridSearchCV(pipeline, candidate_grid, cv=development_splits,
                              scoring='neg_mean_absolute_error', refit=False,
                              n_jobs=1, error_score='raise')
        search.fit(X, y_log)
        models[name] = clone(pipeline).set_params(**search.best_params_).fit(X_train, y_train_log)
        searches[name] = search
        candidate_tables.append(pd.DataFrame(search.cv_results_).assign(model=name))
cv_results = pd.concat(candidate_tables, ignore_index=True)
display(cv_results[['model', 'params', 'mean_test_score', 'std_test_score', 'rank_test_score']]
        .sort_values(['model', 'rank_test_score']))
print({name: search.best_params_ for name, search in searches.items()})

In [ ]:
robust_model = models['robust_huber_lasso']
robust_fit = robust_model.named_steps['regressor']
feature_names = robust_model.named_steps['calendar'].get_feature_names_out()
coefficients = pd.DataFrame({'feature': feature_names, 'standardized_log_coefficient': robust_fit.coef_})
display(coefficients.loc[coefficients.standardized_log_coefficient.ne(0)]
        .sort_values('standardized_log_coefficient', key=abs, ascending=False))
train_diagnostics = visits.iloc[train_index].copy()
train_diagnostics['robust_weight'] = robust_fit.sample_weight_
train_diagnostics['pattern_minutes'] = np.exp(robust_model.predict(X_train))
train_diagnostics['log_residual'] = y_train_log - robust_model.predict(X_train)
positive_residual_band = float(train_diagnostics.log_residual.quantile(0.95))
display(train_diagnostics.robust_weight.describe().to_frame('training fitting weights'))
print(f'Outer iterations: {robust_fit.n_iter_}; converged: {robust_fit.converged_}')
print(f'Training-only positive residual P95: {positive_residual_band:.3f} log units')

## 4. Evaluate the reserved week, retaining the spikes

Report full-distribution MAE, median absolute error, RMSE and log MAE, plus typical/high-residual strata. A high residual means **above the robust model's training-residual P95**, not a verified incident. The same robust-defined mask is applied to all models. Strata use the evaluation outcome for diagnosis only; they are not features, tuning filters or an incident prediction.

The mean per-day median absolute error is another robust pattern diagnostic. Counts matter: aggregate errors can be dominated by many buses on one date. Higher tail RMSE is expected for a seasonality-only model, and does not by itself mean a worse recurring component. All observations, including flagged tails, remain in the full score and saved predictions.

In [ ]:
predictions = visits.iloc[test_index][[
    'trip_instance_key', 'service_day', 'upstream_departure_time', 'arrival_time',
    'available_at', 'traversal_seconds', 'traversal_minutes', 'weekday', 'hour',
]].copy()
for name, model in models.items():
    predictions[name] = np.exp(model.predict(X_test))
predictions['training_median'] = float(np.median(visits.iloc[train_index].traversal_minutes))
predictions['log_residual'] = np.log(predictions.traversal_minutes / predictions.robust_huber_lasso)
predictions['high_residual'] = predictions.log_residual.gt(positive_residual_band)
predictions['signed_residual_seconds'] = predictions.traversal_seconds - 60 * predictions.robust_huber_lasso
predictions['positive_excess_seconds'] = predictions.signed_residual_seconds.clip(lower=0)
metric_rows = []
cohorts = {'all': np.ones(len(predictions), dtype=bool),
           'typical': ~predictions.high_residual, 'high_residual': predictions.high_residual}
for name in [*models, 'training_median']:
    for cohort, mask in cohorts.items():
        subset = predictions.loc[mask]
        if subset.empty:
            print(f'No {cohort} observations; metrics are unavailable for {name}.')
            continue
        actual, predicted = subset.traversal_minutes, subset[name]
        daily_errors = abs(actual - predicted).groupby(subset.service_day).median()
        metric_rows.append({'model': name, 'cohort': cohort, 'visits': len(subset),
            'MAE_minutes': mean_absolute_error(actual, predicted),
            'MedAE_minutes': median_absolute_error(actual, predicted),
            'RMSE_minutes': np.sqrt(mean_squared_error(actual, predicted)),
            'log_MAE': mean_absolute_error(np.log(actual), np.log(predicted)),
            'mean_daily_MedAE_minutes': daily_errors.mean()})
metrics = pd.DataFrame(metric_rows)
display(metrics)
display(predictions.groupby('service_day').agg(visits=('traversal_minutes', 'size'),
        observed_median=('traversal_minutes', 'median'), pattern_median=('robust_huber_lasso', 'median'),
        high_residual_visits=('high_residual', 'sum')))

In [ ]:
evaluation_start = split_summary.iloc[-1].evaluation_start + pd.Timedelta(hours=service_day_start_hour)
evaluation_end = split_summary.iloc[-1].evaluation_end_exclusive + pd.Timedelta(hours=service_day_start_hour)
plot_grid = pd.DataFrame({'upstream_departure_time': pd.date_range(
    evaluation_start.tz_localize(calendar_timezone), evaluation_end.tz_localize(calendar_timezone),
    freq='15min', inclusive='left').tz_convert('UTC')})
plot_clock = plot_grid.upstream_departure_time.dt.tz_convert(calendar_timezone).dt.tz_localize(None)
training_support = train_diagnostics.groupby(['weekday', 'hour']).size()
plot_keys = pd.MultiIndex.from_arrays([plot_clock.dt.dayofweek, plot_clock.dt.hour])
plot_supported = training_support.reindex(plot_keys, fill_value=0).to_numpy() >= minimum_profile_visits
series_figure = go.Figure(go.Scattergl(
    x=predictions.upstream_departure_time.dt.tz_convert(calendar_timezone).dt.tz_localize(None),
    y=predictions.traversal_minutes, mode='markers', name='Observed (spikes retained)',
    marker=dict(size=5, opacity=0.45)))
for name, model in models.items():
    curve = np.exp(model.predict(plot_grid))
    curve[~plot_supported] = np.nan
    series_figure.add_trace(go.Scatter(x=plot_clock, y=curve, name=name, connectgaps=False))
series_figure.update_layout(title='Reserved week: observations and calendar-only traversal patterns',
    xaxis=dict(title='Upstream departure (Vancouver local)', rangeslider=dict(visible=True)),
    yaxis_title='Traversal (minutes)', height=580, legend=dict(orientation='h', y=1.02, yanchor='bottom'),
    margin=dict(t=110, b=100))
series_figure.update_yaxes(fixedrange=False)
figures['reserved_week_patterns'] = series_figure
display(series_figure)

### Training-period time series (in-sample fit)

Compare every training traversal, including spikes, with the two fitted calendar patterns over the full training period. The curves use the same training-support mask, local clock, units and zoom controls as the reserved-week plot. This is an **in-sample fit diagnostic**, not held-out forecast accuracy; embargo and reserved-week observations are not included.

In [ ]:
training_start = train_diagnostics.service_day.min() + pd.Timedelta(hours=service_day_start_hour)
training_grid = pd.DataFrame({'upstream_departure_time': pd.date_range(
    training_start.tz_localize(calendar_timezone).tz_convert('UTC'),
    split_summary.iloc[-1].fit_cutoff_utc, freq='15min', inclusive='left')})
training_clock = training_grid.upstream_departure_time.dt.tz_convert(calendar_timezone).dt.tz_localize(None)
training_keys = pd.MultiIndex.from_arrays([training_clock.dt.dayofweek, training_clock.dt.hour])
training_supported = training_support.reindex(training_keys, fill_value=0).to_numpy() >= minimum_profile_visits
training_figure = go.Figure(layout=series_figure.layout)
training_figure.add_trace(go.Scattergl(
    x=train_diagnostics.upstream_departure_time.dt.tz_convert(calendar_timezone).dt.tz_localize(None),
    y=train_diagnostics.traversal_minutes, mode='markers', name='Observed training traversals (spikes retained)',
    marker=dict(size=5, opacity=0.45)))
for name, model in models.items():
    curve = np.exp(model.predict(training_grid))
    curve[~training_supported] = np.nan
    training_figure.add_trace(go.Scatter(x=training_clock, y=curve, name=name, connectgaps=False))
training_figure.update_layout(title='Training period: observations and calendar-only patterns (in-sample fit)')
figures['training_period_patterns'] = training_figure
display(training_figure)

## 5. Inspect the learned repeating pattern and spike sensitivity

Draw a representative local week, masking weekday/hour cells with fewer than `minimum_profile_visits` training observations. Missing support is not zero traversal. The support table also reports distinct training days; curves in rarely observed cells remain uncertain. These are pooled patterns of observed buses, not time-weighted general road traffic.

Then freeze the selected parameters, inject artificial **8x positive spikes into 5% of training labels**, refit, and measure changes on the same supported calendar grid. This is a sensitivity experiment, not new parameter selection or proof against every contamination mechanism. Robust loss bounds amplitude influence; concentrated incidents, a majority of abnormal journeys at a particular clock time, or repeated slowdowns can still shift the estimated pattern. Small delta may also attenuate genuine peaks, so compare supported profiles rather than assuming every suppressed peak is an incident.

In [ ]:
profile_grid = pd.DataFrame({'upstream_departure_time': pd.date_range(
    representative_monday, periods=7 * 96, freq='15min', tz=calendar_timezone).tz_convert('UTC')})
profile_clock = profile_grid.upstream_departure_time.dt.tz_convert(calendar_timezone).dt.tz_localize(None)
profile_keys = pd.MultiIndex.from_arrays([profile_clock.dt.dayofweek, profile_clock.dt.hour])
profile_supported = training_support.reindex(profile_keys, fill_value=0).to_numpy() >= minimum_profile_visits
profile = pd.DataFrame({'weekday': profile_clock.dt.day_name(),
                        'hour': profile_clock.dt.hour + profile_clock.dt.minute / 60})
for name, model in models.items():
    profile[name] = np.exp(model.predict(profile_grid))
    profile.loc[~profile_supported, name] = np.nan
support_table = train_diagnostics.groupby(['weekday', 'hour']).agg(
    visits=('traversal_minutes', 'size'), distinct_days=('service_day', 'nunique'),
    observed_median_minutes=('traversal_minutes', 'median'),
    fitted_median_minutes=('pattern_minutes', 'median'))
display(support_table)
profile_figure = px.line(profile, x='hour', y='robust_huber_lasso', color='weekday',
    category_orders={'weekday': ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']},
    title='Robust learned daily profiles (only training-supported hours)',
    labels={'hour': 'Vancouver local hour', 'robust_huber_lasso': 'Typical traversal (minutes)'}, height=520)
profile_figure.update_traces(connectgaps=False)
figures['robust_daily_profiles'] = profile_figure
display(profile_figure)

In [ ]:
rng = np.random.default_rng(random_state)
injected_index = rng.choice(len(y_train_log), size=max(1, int(spike_fraction * len(y_train_log))), replace=False)
contaminated_y_log = y_train_log.copy()
contaminated_y_log[injected_index] += np.log(spike_multiplier)
sensitivity_rows = []
with threadpool_limits(limits=1):
    for name, model in models.items():
        contaminated_model = clone(model).fit(X_train, contaminated_y_log)
        before = np.exp(model.predict(profile_grid))[profile_supported]
        after = np.exp(contaminated_model.predict(profile_grid))[profile_supported]
        movement = abs(after - before)
        sensitivity_rows.append({'model': name, 'injected_training_visits': len(injected_index),
            'supported_grid_points': len(movement), 'mean_pattern_shift_minutes': movement.mean(),
            'p95_pattern_shift_minutes': np.quantile(movement, 0.95),
            'max_pattern_shift_minutes': movement.max()})
sensitivity = pd.DataFrame(sensitivity_rows)
display(sensitivity)
residual_figure = px.scatter(predictions.assign(departure_display=predictions.upstream_departure_time
                            .dt.tz_convert(calendar_timezone).dt.tz_localize(None)),
    x='departure_display', y='signed_residual_seconds', color='high_residual',
    hover_data=['trip_instance_key', 'positive_excess_seconds'],
    title='Residuals left for a separate spike/dynamic model',
    labels={'departure_display': 'Vancouver local departure', 'signed_residual_seconds': 'Observed - pattern (seconds)'})
residual_figure.add_hline(y=0, line_dash='dash')
figures['remaining_residuals'] = residual_figure
display(residual_figure)

### Results from the pinned snapshot (original traversal experiment)

The executed run retained **3,156** full-day traversals. The final fit used 2,701 visits on 50 observed service days; the reserved 3-9 November week contained 391 visits. Development selected daily order 6, weekly order 3, and alpha 0.003 for Huber-Lasso (13 outer iterations, converged). The ordinary Lasso selected alpha 0.01 with the same orders.

Injecting 8x spikes into 5% of training labels moved the robust supported calendar curve by **0.088 minutes on average**, versus **0.816 minutes** for ordinary Lasso: about **9.3x less movement**. The corresponding P95 movements were 0.232 and 1.596 minutes. This supports spike-amplitude resistance under the tested contamination, not complete incident immunity.

**Do not confuse resistance with predictive improvement:** reserved-week MAE was 1.243 minutes for Huber-Lasso, 1.250 for ordinary Lasso, and **1.133 for the global training median**. The median also had lower log MAE. The robust seasonal model therefore did **not** beat the constant benchmark on this week. Observed daily medians were generally below the fitted patterns, consistent with a possible level shift that a deliberately trend-free calendar component cannot track. Keep this as an exploratory recurring-pattern model, not a demonstrated forecast improvement; retain the median comparator and verify on untouched future weeks before deployment.

## 6. Earliest-feed implied traversal versus harmonic regression

Compare **traversal time, not downstream delay**, for the same segment 4 visits in the reserved week. The early-feed comparator uses the first downstream publication and the latest upstream publication available **at or before that same origin**. Missing selected values are not replaced with later or older reports. Exact dated journeys, endpoint IDs and adjacent numeric stop sequences are required.

Infer scheduled arrivals from those early reports only. With upstream/downstream denoted by $u/d$, early/latest reports by $0/L$, arrival time by $A$, upstream departure by $P$, and arrival delay by $D$, the early scheduled allowance and zero-dwell traversal estimate are:

$$
S_0=(A_{d,0}-D_{d,0})-(A_{u,0}-D_{u,0}),\qquad
\widehat T_{\mathrm{early}}=S_0+D_{d,0}-D_{u,0}.
$$

This simplifies to $A_{d,0}-A_{u,0}$: the early predicted arrival-to-arrival interval, interpreted as road traversal because **predicted dwell is assumed zero**. It is an inversion of the feed's endpoint forecasts, not a separately trained dynamics model. Finite negative early predictions are flagged and scored, not clipped or removed.

The harmonic pipeline remains frozen. Its input clock here is the early-implied scheduled upstream time $A_{u,0}-D_{u,0}$, with zero dwell making scheduled arrival and departure coincide. Unlike the original traversal evaluation above, **no final observed departure time enters this prediction**. The traversal regressor has only calendar features, although this as-of timetable clock is recovered from early feed fields rather than independently verified historical GTFS.

For the reference only, use latest endpoint reports and **observed upstream dwell** $W_L=P_{u,L}-A_{u,L}$:

$$
T_{\mathrm{reference}}=S_L+D_{d,L}-D_{u,L}-W_L=A_{d,L}-P_{u,L}.
$$

Latest schedules/delays and observed dwell never enter either early prediction. The accounting equality is a calculation check, not independent ground truth. For reserved-week performance, require origins after the harmonic fit cutoff and before upstream departure. Both methods are scored on exactly the same eligible visits. The additional training figures are explicitly in-sample diagnostics, not out-of-time scores.

Both comparison time series have a second row showing how many minutes before the **early-implied scheduled downstream arrival at Queensway** the earliest-feed prediction was published. This is scheduled downstream arrival minus publication origin, not time to actual upstream departure or latest arrival. Negative values mean publication after the scheduled arrival. The two method-specific scatter figures use **prediction on X** and **latest-reference actual traversal on Y**, with separate training and reserved-week panels and 1:1 lines. Do not tune the harmonic model from these figures; the same-cache reserved week remains exploratory.

In [ ]:
early_comparison = prepare_early_traversal_comparison(
    dataset.trip_updates, selected_segment, route_id=route_id, timezone=calendar_timezone,
)
comparison_keys = ['trip_instance_key', 'stop_id', 'stop_sequence']
comparison_candidates = visits.iloc[test_index][comparison_keys + ['service_day']].merge(
    early_comparison.observations, on=comparison_keys, how='left', validate='one_to_one',
)
comparison_reasons = comparison_candidates.comparison_exclusion_reason.fillna('missing_latest_reference')
comparison_reasons.loc[comparison_reasons.eq('included') & comparison_candidates.origin_time.lt(
    split_summary.iloc[-1].fit_cutoff_utc)] = 'origin_before_harmonic_fit'
comparison_candidates['comparison_exclusion_reason'] = comparison_reasons
reserved_comparison_audit = comparison_reasons.value_counts().rename_axis('reason').reset_index(name='count')
reserved_comparison_audit['stage'] = 'reserved_week_comparison'
comparison_audit = pd.concat([early_comparison.audit, reserved_comparison_audit], ignore_index=True)
comparison = comparison_candidates.loc[comparison_reasons.eq('included')].copy().reset_index(drop=True)
if comparison.empty:
    raise ValueError('No paired reserved-week traversal comparisons; inspect comparison_audit.')
comparison_X = pd.DataFrame({'upstream_departure_time': comparison.early_scheduled_upstream_at})
comparison['harmonic_traversal_seconds'] = 60 * np.exp(robust_model.predict(comparison_X))
if not np.isfinite(comparison.harmonic_traversal_seconds).all():
    raise ValueError('Harmonic traversal comparison contains nonfinite predictions.')
display(comparison_audit)
display(comparison[['early_scheduled_arrival_lead_minutes', 'origin_lead_minutes', 'early_allowance_seconds', 'dwell_seconds',
                    'schedule_revision_seconds']].describe())
print(f'Paired comparisons: {len(comparison):,}/{len(comparison_candidates):,} reserved-week visits.')
print(f'Finite nonpositive early predictions retained: {comparison.early_nonpositive_traversal.sum()}')

In [ ]:
comparison_methods = {'early_feed': 'early_implied_traversal_seconds',
                      'harmonic': 'harmonic_traversal_seconds'}
comparison_metric_rows = []
for name, column in comparison_methods.items():
    error_minutes = (comparison[column] - comparison.reference_traversal_seconds) / 60
    comparison[f'{name}_error_minutes'] = error_minutes
    comparison[f'{name}_absolute_error_minutes'] = abs(error_minutes)
    comparison_metric_rows.append({'method': name, 'visits': len(comparison),
        'MAE_minutes': abs(error_minutes).mean(), 'MedAE_minutes': abs(error_minutes).median(),
        'RMSE_minutes': np.sqrt((error_minutes ** 2).mean()), 'bias_minutes': error_minutes.mean()})
comparison_metrics = pd.DataFrame(comparison_metric_rows)
early_error = comparison.early_feed_absolute_error_minutes
harmonic_error = comparison.harmonic_absolute_error_minutes
tied_errors = np.isclose(early_error, harmonic_error, rtol=0, atol=1e-9)
comparison_wins = pd.DataFrame([{
    'harmonic_lower_error': int(((harmonic_error < early_error) & ~tied_errors).sum()),
    'early_feed_lower_error': int(((early_error < harmonic_error) & ~tied_errors).sum()),
    'ties': int(tied_errors.sum()), 'paired_visits': len(comparison),
}])
comparison_daily = comparison.groupby('service_day').agg(
    visits=('reference_traversal_seconds', 'size'),
    early_feed_MAE_minutes=('early_feed_absolute_error_minutes', 'mean'),
    harmonic_MAE_minutes=('harmonic_absolute_error_minutes', 'mean'),
    median_origin_lead_minutes=('origin_lead_minutes', 'median'),
).reset_index()
display(comparison_metrics)
display(comparison_wins)
display(comparison_daily)

### Training-period comparison (in-sample diagnostic)

Use the same early-feed pairing, zero-dwell assumption and early-implied scheduled harmonic clock for the visits used in the final harmonic fit. Training origins naturally precede that fit cutoff; do not apply the reserved-week origin-after-fit guard to this diagnostic. Latest-reference eligibility and predeparture checks still apply, and excluded training visits are audited. These figures show fit, not independent forecasting performance.

In [ ]:
training_comparison_candidates = visits.iloc[train_index][comparison_keys + ['service_day']].merge(
    early_comparison.observations, on=comparison_keys, how='left', validate='one_to_one',
)
training_comparison_reasons = training_comparison_candidates.comparison_exclusion_reason.fillna('missing_latest_reference')
training_comparison_candidates['comparison_exclusion_reason'] = training_comparison_reasons
training_comparison_audit = training_comparison_reasons.value_counts().rename_axis('reason').reset_index(name='count')
training_comparison_audit['stage'] = 'training_comparison'
training_comparison = training_comparison_candidates.loc[training_comparison_reasons.eq('included')].copy().reset_index(drop=True)
if training_comparison.empty:
    raise ValueError('No paired training traversal comparisons; inspect training_comparison_audit.')
training_comparison_X = pd.DataFrame({'upstream_departure_time': training_comparison.early_scheduled_upstream_at})
training_comparison['harmonic_traversal_seconds'] = 60 * np.exp(robust_model.predict(training_comparison_X))
if not np.isfinite(training_comparison.harmonic_traversal_seconds).all():
    raise ValueError('Training traversal comparison contains nonfinite harmonic predictions.')
display(training_comparison_audit)
print(f'Paired training comparisons (in-sample): {len(training_comparison):,}/{len(training_comparison_candidates):,} visits.')

In [ ]:
training_comparison_figure = make_traversal_comparison_timeseries(
    training_comparison, title='Training period: traversal comparison (in-sample fit)',
    timezone=calendar_timezone,
)
figures['early_feed_training_traversal_comparison'] = training_comparison_figure
display(training_comparison_figure)

In [ ]:
comparison_series_figure = make_traversal_comparison_timeseries(
    comparison,
    title='Reserved week: early-feed versus harmonic traversal predictions',
    timezone=calendar_timezone,
)
figures['early_feed_traversal_comparison'] = comparison_series_figure
display(comparison_series_figure)

In [ ]:
figures.pop('early_feed_traversal_error_comparison', None)
for name, column, label in [
    ('early_feed_traversal_prediction_scatter', 'early_implied_traversal_seconds', 'Earliest feed (zero dwell)'),
    ('harmonic_traversal_prediction_scatter', 'harmonic_traversal_seconds', 'Harmonic regression'),
]:
    figure = make_traversal_prediction_scatter(
        training_comparison, comparison, prediction_column=column, method_name=label,
    )
    figures[name] = figure
    display(figure)

### Results for the early-origin comparison

All **391** reserved-week visits passed the paired comparison guards. Earliest-feed implied traversal had **MAE 1.022 minutes** and **RMSE 1.471 minutes**; harmonic regression at the early-implied scheduled clock had **MAE 1.248 minutes** and **RMSE 1.811 minutes**. Earliest feed had lower absolute error on **212 visits**, harmonic regression on **179**, with no ties. Thus the earliest-feed comparator performed better overall on this week, although not on every visit.

These harmonic scores differ slightly from the original traversal experiment because the prediction clock is now available at the early origin rather than taken from final departure. The harmonic coefficients were not refitted. Results remain exploratory, reference values are final-feed proxies, and the early comparator's zero-dwell approximation is deliberately not applied to truth.

## 7. Persist the experiment and keep the components separate

Each run creates a **new, non-overwriting** directory under `data/segment_harmonic_models/`, containing fitted pipelines (including transformer/scaler), selected parameters, dataset checksums, split cutoffs, scores, coefficients, training weights, calendar profiles, sensitivity results, predictions/residuals, paired early-feed comparisons and audits for training/reserved visits, reserved-week scores, and eight offline HTML figures. Inputs and other notebooks' exports are never overwritten. Pickles must only be loaded from trusted sources. Predictions from a loaded pipeline are in **log minutes**; apply `np.exp` exactly once.

The final export cell also saves all eight plots as self-contained HTML under `notebooks/visuals/segment_4_harmonic_traversal_regression/`, matching this notebook's name. The two predicted-versus-reference scatter figures replace the earlier error-versus-error plot; archived experiment bundles remain unchanged. No model fitting is needed to re-export figures already created in the kernel.

`positive_excess_seconds` is a retrospective excess, not a forecast or an incident label. Future composition can add a separately predicted positive excess to the recurring component. Training that second component must construct calendar residuals **out of fold**, and any dynamic inputs must be already completed and received at the forecast origin. Do not train it from this notebook's in-sample calendar residuals as if they were independent predictions. A new untouched future period is needed before claiming prospective skill.

In [ ]:
metadata = {
    'dataset_id': dataset_id, 'source_files': manifest['files'],
    'route_id': route_id, 'direction_id': direction_id, 'segment_id': segment_id,
    'stop_pair': list(expected_stop_pair), 'calendar_timezone': calendar_timezone,
    'service_day_start_hour': service_day_start_hour, 'target': 'log(traversal_minutes)',
    'huber_delta': huber_delta, 'alpha_grid': alpha_grid, 'harmonic_orders': harmonic_orders,
    'selection_score': 'development log MAE', 'selected_parameters': {
        name: search.best_params_ for name, search in searches.items()},
    'positive_residual_training_p95': positive_residual_band,
    'spike_sensitivity': {'fraction': spike_fraction, 'multiplier': spike_multiplier, 'seed': random_state},
    'boundary_rows_excluded': boundary_rows, 'minimum_profile_visits': minimum_profile_visits,
    'robust_converged': bool(robust_fit.converged_),
    'early_traversal_comparison': {'early_dwell_seconds': 0,
        'schedule_source': 'origin_feed_implied', 'harmonic_clock': 'early_scheduled_upstream_at',
        'reference': 'latest departure-to-arrival traversal; observed dwell removed',
        'paired_reserved_visits': len(comparison), 'paired_training_visits': len(training_comparison),
        'lead_time': 'early-implied scheduled downstream arrival minus publication origin'},
    'splits': json.loads(split_summary.to_json(orient='records', date_format='iso')),
    'validation_caveat': 'Exploratory same-cache temporal evaluation; EDA has seen the full cache.',
}
experiment_dir = save_harmonic_experiment(output_root, models=models, predictions=predictions,
                                         cv_results=cv_results, metadata=metadata)
for name, table in {'metrics': metrics, 'sensitivity': sensitivity, 'coefficients': coefficients,
                    'training_diagnostics': train_diagnostics, 'profiles': profile,
                    'support': support_table.reset_index(), 'splits': split_summary,
                    'traversal_comparison': comparison, 'traversal_comparison_candidates': comparison_candidates,
                    'traversal_comparison_audit': comparison_audit, 'traversal_comparison_metrics': comparison_metrics,
                    'traversal_comparison_wins': comparison_wins, 'traversal_comparison_daily': comparison_daily,
                    'training_traversal_comparison': training_comparison,
                    'training_traversal_comparison_candidates': training_comparison_candidates,
                    'training_traversal_comparison_audit': training_comparison_audit}.items():
    table.to_csv(experiment_dir / f'{name}.csv', index=False)
for name, figure in figures.items():
    figure.write_html(experiment_dir / f'{name}.html', full_html=True, include_plotlyjs=True)
print(f'Saved experiment: {experiment_dir}')

In [ ]:
figure_output_dir.mkdir(parents=True, exist_ok=True)
for name, figure in figures.items():
    figure.write_html(figure_output_dir / f'{name}.html', full_html=True, include_plotlyjs=True)
(figure_output_dir / 'early_feed_traversal_error_comparison.html').unlink(missing_ok=True)
print(f'Saved {len(figures)} notebook plots: {figure_output_dir}')